# Error analysis on Stage-2 BiLSTM model
Hypothesis checks on the Stage-2 test predictions. 
- Setup cells are copied from main.ipynb on bilstm branch
- Skipped training
- Saved checkpoint (stage2_ensemble_best.pt) is loaded
- GloVe is left empty as the word vectors are already stored in the checkpoint.

## 1. Imports and configuration

In [1]:
import os, re, math, copy, time, zipfile, io, collections
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torchcrf import CRF

# Loads python packages for natural language processing
import nltk
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

True

Loading the dataset file located in data folder. If it doesn't exist please import it

Additionally, GloVe 6B comes in four sizes (50,100,200,300)

Largest size is choosen for best performance

In [2]:
DATASET_FILE = "en-US.jsonl"

# Load the dataset file
DATA_FOLDER = "data"
if not os.path.exists(os.path.join(DATA_FOLDER, DATASET_FILE)):
    raise FileNotFoundError(f"Could not find {os.path.join(DATA_FOLDER, DATASET_FILE)}")
print("Using dataset at:", os.path.join(DATA_FOLDER, DATASET_FILE))

SEED = 4002
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Configuration parameters for the model
EMB_DIM_WANTED = 300   # GloVe 6B dimension
HIDDEN         = 256
NUM_LAYERS     = 2
DROPOUT        = 0.3
LR             = 1e-3
WEIGHT_DECAY   = 0.01
LABEL_SMOOTH   = 0.1
WORD_DROPOUT   = 0.25
BATCH_SIZE     = 32
NUM_EPOCHS     = 40
PATIENCE       = 8
GRAD_CLIP      = 5.0

print("Device:", device)

Using dataset at: data\en-US.jsonl
Device: cuda


## 2. Dataset and splits

Load `en-US.jsonl` and split it by the official `partition` field: **training** = `train` (11,514),
**validation** = `dev` (2,033), **test** = `test` (2,974). The input is `utt`. The **Task 1** label is
`intent` and the **Task 2** label is parsed from `annot_utt`.

In [3]:
df = pd.read_json(os.path.join(DATA_FOLDER, DATASET_FILE), lines=True)

assert (df["partition"] == "train").sum() == 11514
assert (df["partition"] == "dev").sum()   == 2033
assert (df["partition"] == "test").sum()  == 2974
print("Dataset shape:", df.shape, "| split sizes asserted")

train_df = df[df["partition"] == "train"].copy()
val_df   = df[df["partition"] == "dev"].copy()
test_df  = df[df["partition"] == "test"].copy()

x_train, y_train = train_df["utt"].tolist(), train_df["intent"].tolist()
x_val,   y_val   = val_df["utt"].tolist(),   val_df["intent"].tolist()
print("Training examples:  ", len(train_df))
print("Validation examples:", len(val_df))
print("Test examples:      ", len(test_df))

Dataset shape: (16521, 8) | split sizes asserted
Training examples:   11514
Validation examples: 2033
Test examples:       2974


## 3. Word vectors

(Skipped) Word vectors are loaded from the saved checkpoint, so `glove.6B.zip` isn't needed.

In [4]:
# GloVe values are loaded from the saved checkpoint, so no need for glove.6B.zip here
glove = {}

## 4. Tokenization, vocabulary and lemmatization

Utterances are lowercased and split into word and punctuation tokens. Unseen words can optionally fall back to their WordNet lemma before mapping to `<UNK>`.

In [5]:
token_pattern = re.compile(r'\w+|[^\w\s]')
PAD_TOKEN, UNK_TOKEN = "<PAD>", "<UNK>"
PAD_ID, UNK_ID = 0, 1
SLOT_PAD_ID = -100


def tokenize_with_offsets(text):
    return [(m.group().lower(), m.start(), m.end()) for m in token_pattern.finditer(text)]


def tokenize(text):
    return [t for t, _, _ in tokenize_with_offsets(text)]


lemmatizer = WordNetLemmatizer()


def lemmatize_token(token):
    # Try the noun form first (plurals), then the verb form (tenses)
    lemma = lemmatizer.lemmatize(token, pos='n')
    if lemma == token:
        lemma = lemmatizer.lemmatize(token, pos='v')
    return lemma


def token_to_id(token, word_to_id, use_lemma):
    # Known words keep their own ID; only unseen words fall back to their lemma
    if token in word_to_id:
        return word_to_id[token]
    if use_lemma:
        lemma = lemmatize_token(token)
        if lemma in word_to_id:
            return word_to_id[lemma]
    return UNK_ID

## 5. Label preparation: Task 1 intents and Task 2 slot alignment

- **Task 1:** each `intent` string is mapped to a class ID (60 classes).
- **Task 2:** the `[slot type : span]` annotations in `annot_utt` are aligned to the tokens as BIO tags
  (111 tags = `O` + B-/I- for each of the 55 slot types).

In [6]:
slot_pattern = re.compile(r'\[\s*([^:\[\]]+?)\s*:\s*([^\[\]]*?)\]')


def parse_annotated_utterance(utt, annot_utt):
    spans, plain_text, last_end = [], '', 0
    for match in slot_pattern.finditer(annot_utt):
        plain_text += annot_utt[last_end:match.start()]
        slot_type, slot_value = match.group(1).strip(), match.group(2)
        start = len(plain_text)
        plain_text += slot_value
        spans.append({'type': slot_type, 'start': start,
                      'end': len(plain_text), 'text': slot_value})
        last_end = match.end()
    plain_text += annot_utt[last_end:]
    if plain_text != utt:
        raise ValueError(f"Slot annotation does not match utterance: {utt!r}")
    return spans


def make_aligned_tokens(utt, spans):
    tags, offsets, prev_slot = [], [], None
    for word, start, end in tokenize_with_offsets(utt):
        current_slot = None
        for i, span in enumerate(spans):
            if start < span['end'] and end > span['start']:
                if start < span['start'] or end > span['end']:
                    raise ValueError(f"A token cuts across a slot boundary: {utt!r}")
                current_slot = i
        if current_slot is None:
            tags.append('O')
        else:
            prefix = 'I' if current_slot == prev_slot else 'B'
            tags.append(f"{prefix}-{spans[current_slot]['type']}")
        offsets.append((start, end))
        prev_slot = current_slot
    return tags, offsets


def prepare_joint_records(split_df):
    records = []
    for row_index, row in split_df.iterrows():
        text = row['utt']
        spans = parse_annotated_utterance(text, row['annot_utt'])
        tags, offsets = make_aligned_tokens(text, spans)
        tokens = tokenize(text)
        if len(tokens) != len(tags):
            raise ValueError(f"Token and slot lengths differ at row {row_index}")
        records.append({'source_row': int(row_index), 'utt': text, 'tokens': tokens,
                        'offsets': offsets, 'gold_spans': spans,
                        'slot_tags': tags, 'intent': row['intent']})
    return records


train_records = prepare_joint_records(train_df)
val_records   = prepare_joint_records(val_df)
test_records  = prepare_joint_records(test_df)

intents = sorted(set(y_train))
intent_to_id = {v: i for i, v in enumerate(intents)}
num_classes = len(intent_to_id)

training_slot_types = sorted({s['type'] for r in train_records for s in r['gold_spans']})
slot_tags = ['O']
for t in training_slot_types:
    slot_tags.extend([f'B-{t}', f'I-{t}'])
slot_to_id = {tag: i for i, tag in enumerate(slot_tags)}
id_to_slot = {i: tag for tag, i in slot_to_id.items()}
num_slot_labels = len(slot_to_id)

max_len = max(64, max(len(r['tokens']) for r in train_records))
assert num_classes == 60 and num_slot_labels == 111
print('Intent classes:', num_classes, '| BIO labels:', num_slot_labels, '| max_len:', max_len)

Intent classes: 60 | BIO labels: 111 | max_len: 64


## 6. Dataset, loaders and the helpers the comparison needs

In [7]:
class JointNLUDataset(Dataset):
    def __init__(self, records, word_to_id, intent_to_id, slot_to_id, max_len, use_lemma=False):
        self.records = records
        self.input_ids, self.attention_masks = [], []
        self.intent_ids, self.slot_ids = [], []
        for row in records:
            words = [token_to_id(w, word_to_id, use_lemma) for w in row['tokens']]
            slots = [slot_to_id[tag] for tag in row['slot_tags']]
            if len(words) > max_len:
                raise ValueError(f"Sentence has {len(words)} tokens, limit is {max_len}")
            padding = max_len - len(words)
            self.input_ids.append(torch.tensor(words + [PAD_ID] * padding))
            self.attention_masks.append(
                torch.tensor([1] * len(words) + [0] * padding, dtype=torch.bool))
            self.intent_ids.append(torch.tensor(intent_to_id[row['intent']]))
            self.slot_ids.append(torch.tensor(slots + [SLOT_PAD_ID] * padding))

    def __len__(self):
        return len(self.records)

    def __getitem__(self, i):
        row = self.records[i]
        return {'input_ids': self.input_ids[i], 'attention_mask': self.attention_masks[i],
                'intent_labels': self.intent_ids[i], 'slot_labels': self.slot_ids[i],
                'utt': row['utt'], 'offsets': row['offsets'],
                'gold_spans': row['gold_spans'], 'source_row': row['source_row']}


def joint_collate_fn(samples):
    batch = {}
    for key in ('input_ids', 'attention_mask', 'intent_labels', 'slot_labels'):
        batch[key] = torch.stack([item[key] for item in samples])
    for key in ('utt', 'offsets', 'gold_spans', 'source_row'):
        batch[key] = [item[key] for item in samples]
    return batch


def build_vocab(records):
    # Vocabulary from the given training records only
    vocab = {PAD_TOKEN: PAD_ID, UNK_TOKEN: UNK_ID}
    for word in sorted({t for row in records for t in row['tokens']}):
        vocab[word] = len(vocab)
    return vocab


def build_embedding_matrix(vocab):
    matrix_rng = np.random.default_rng(SEED)
    matrix = matrix_rng.normal(0, 0.1, (len(vocab), EMB_DIM_WANTED)).astype(np.float32)
    matrix[PAD_ID] = 0
    for word, idx in vocab.items():
        if word in glove:
            matrix[idx] = glove[word]
    return matrix


def make_loader(records, vocab, use_lemma=False, shuffle=False, batch_size=None):
    dataset = JointNLUDataset(records, vocab, intent_to_id, slot_to_id, max_len,
                              use_lemma=use_lemma)
    generator = torch.Generator().manual_seed(SEED) if shuffle else None
    return DataLoader(dataset, batch_size=batch_size or BATCH_SIZE, shuffle=shuffle,
                      collate_fn=joint_collate_fn, generator=generator)


def lemmatize_records(records):
    # Lemmatize model input tokens only; utt, offsets and gold spans stay original,
    # so slot scoring on character spans is unaffected.
    return [{**row, 'tokens': [lemmatize_token(t) for t in row['tokens']]} for row in records]


def word_frequencies(records, vocab):
    counts = collections.Counter(t for row in records for t in row['tokens'])
    freq = torch.zeros(len(vocab))
    for w, i in vocab.items():
        freq[i] = counts.get(w, 0)
    return freq.to(device)


word_to_id = build_vocab(train_records)
embedding_matrix = build_embedding_matrix(word_to_id)
print("Vocabulary size:", len(word_to_id))
print("Training tokens missing from GloVe:",
      sum(w not in glove for w in word_to_id if w not in (PAD_TOKEN, UNK_TOKEN)))

Vocabulary size: 5146
Training tokens missing from GloVe: 5144


## 7. Stage-1 model and shared evaluation functions (Task 1 and Task 2)

- **Inputs:** lower-cased regex tokens mapped to word IDs (training-split vocabulary; unseen words fall back
  to their WordNet lemma when the "with fallback" variant is used).
- **Embedding:** 300d, initialised from GloVe 6B (random for the 181 training words GloVe lacks), fine-tuned.
- **Encoder:** dropout → 2-layer BiLSTM, 256 units per direction, packed sequences.
- **Task 1 output (intent classification):** concat(max-pool, mean-pool) of BiLSTM states → linear → 60 intent logits.
- **Task 2 output (slot filling):** per-token linear emissions over 111 BIO tags → CRF (Viterbi decoding).
- **Loss (joint):** Task 1 intent cross-entropy (label smoothing 0.1) + Task 2 CRF negative log-likelihood.
- **Optimiser:** AdamW (lr 1e-3, weight decay 0.01), cosine annealing, gradient clipping 5,
  frequency-based word dropout to `<UNK>` (α = 0.25), batch size 32, ≤ 40 epochs with early stopping.

**Shared scoring (used by both stages):** predicted BIO tags are converted to character spans in the original `utt`.

- **Task 1:** intent accuracy = 100 × correct / utterances
- **Task 2:** slot micro-F1 = 100 × 2TP / (2TP + FP + FN) over exact `(slot type, span)` matches pooled across utterances

In [8]:
class JointBiLSTM(nn.Module):
    def __init__(self, embedding_matrix, num_intents, num_slots):
        super().__init__()
        vocab_size, d_model = embedding_matrix.shape
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.embedding.weight.data.copy_(torch.as_tensor(embedding_matrix))
        self.lstm = nn.LSTM(d_model, HIDDEN, NUM_LAYERS, batch_first=True,
                            bidirectional=True, dropout=DROPOUT)
        self.dropout = nn.Dropout(DROPOUT)
        self.slot_classifier = nn.Linear(2 * HIDDEN, num_slots)
        self.intent_classifier = nn.Linear(4 * HIDDEN, num_intents)
        self.crf = CRF(num_slots, batch_first=True)

    def forward(self, input_ids, attention_mask):
        lengths = attention_mask.sum(dim=1)
        x = self.dropout(self.embedding(input_ids))
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True,
                                      enforce_sorted=False)
        h, _ = self.lstm(packed)
        h, _ = pad_packed_sequence(h, batch_first=True, total_length=input_ids.size(1))
        h = self.dropout(h)
        m = attention_mask.unsqueeze(-1)
        max_pool = h.masked_fill(~m, float('-inf')).max(dim=1).values
        mean_pool = (h * m).sum(dim=1) / m.sum(dim=1).clamp(min=1)
        return self.intent_classifier(torch.cat([max_pool, mean_pool], dim=-1)), \
               self.slot_classifier(h)


def tags_to_spans(tags):
    spans, cur, start = set(), None, 0
    for i, t in enumerate(tags):
        if t == 'O':
            if cur:
                spans.add((cur, start, i))
                cur = None
        else:
            prefix, slot_type = t[:1], t[2:]
            if prefix == 'B' or cur != slot_type:
                if cur:
                    spans.add((cur, start, i))
                cur, start = slot_type, i
    if cur:
        spans.add((cur, start, len(tags)))
    return spans


def spans_to_char(spans, offsets):
    # Scoring on character spans keeps the metric independent of tokenization
    return {(t, offsets[s][0], offsets[e - 1][1]) for t, s, e in spans}


def gold_char_spans(gold_spans):
    return {(s['type'], s['start'], s['end']) for s in gold_spans}


def intent_accuracy(predicted, gold):
    """100 x correct intent predictions / number of utterances."""
    return 100 * sum(p == g for p, g in zip(predicted, gold)) / len(gold)


def slot_micro_f1(predicted_spans, gold_spans):
    """F1 on exact (slot type, span) matches pooled over utterances: 100 x 2TP / (2TP + FP + FN)."""
    tp = fp = fn = 0
    for pred, gold in zip(predicted_spans, gold_spans):
        tp += len(gold & pred)
        fp += len(pred - gold)
        fn += len(gold - pred)
    return 0.0 if tp == 0 else 100 * 2 * tp / (2 * tp + fp + fn)


assert tags_to_spans(['O','O','B-time','I-time']) == {('time', 2, 4)}
assert tags_to_spans(['I-time','I-time','O','I-date','B-date']) == \
       {('time',0,2), ('date',3,4), ('date',4,5)}
assert intent_accuracy(['a', 'b', 'c', 'd'], ['a', 'b', 'x', 'y']) == 50
assert slot_micro_f1([{('time', 0, 4), ('date', 5, 9)}], [{('time', 0, 4)}]) == 100 * 2 / 3
print('scoring self-tests passed')

scoring self-tests passed


### Training and evaluation functions

- `unpack`: moves a batch to the device, trims padding-only columns and (during training) applies word dropout to `<UNK>`.
- `compute_loss`: joint loss = Task 1 intent cross-entropy + Task 2 CRF negative log-likelihood.
- `train_one_epoch`: one pass over the training data with gradient clipping and a scheduler step per batch.
- `evaluate`: returns the loss, Task 1 intent accuracy and Task 2 slot micro-F1 on a given split.

In [9]:
def unpack(batch, freq=None, training=False):
    ids  = batch['input_ids'].to(device)
    mask = batch['attention_mask'].to(device)
    slots = batch['slot_labels'].to(device)
    T = int(mask.sum(dim=1).max())              # trim padding-only columns
    ids, mask, slots = ids[:, :T], mask[:, :T], slots[:, :T]
    if training and WORD_DROPOUT > 0 and freq is not None:
        p = WORD_DROPOUT / (WORD_DROPOUT + freq[ids])
        hit = (torch.rand_like(p) < p) & mask & (ids > UNK_ID)
        ids = torch.where(hit, torch.full_like(ids, UNK_ID), ids)
    return ids, mask, slots, batch['intent_labels'].to(device)


def compute_loss(model, intent_logits, emissions, mask, slots, intent):
    return F.cross_entropy(intent_logits, intent, label_smoothing=LABEL_SMOOTH) \
           - model.crf(emissions, slots.clamp(min=0), mask=mask, reduction='mean')


def train_one_epoch(model, loader, optimizer, scheduler, freq):
    model.train()
    total_loss, total = 0.0, 0
    for batch in loader:
        ids, mask, slots, intent = unpack(batch, freq, training=True)
        optimizer.zero_grad()
        intent_logits, emissions = model(ids, mask)
        loss = compute_loss(model, intent_logits, emissions, mask, slots, intent)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        if scheduler:
            scheduler.step()
        total_loss += loss.item() * ids.size(0)
        total += ids.size(0)
    return total_loss / total


def evaluate(model, loader):
    '''Returns (loss, intent accuracy, slot micro-F1) using the shared scoring functions.'''
    model.eval()
    total_loss, total = 0.0, 0
    pred_intents, gold_intents, pred_spans, gold_spans = [], [], [], []
    with torch.no_grad():
        for batch in loader:
            ids, mask, slots, intent = unpack(batch)
            intent_logits, emissions = model(ids, mask)
            total_loss += compute_loss(model, intent_logits, emissions, mask,
                                       slots, intent).item() * ids.size(0)
            total += ids.size(0)
            pred_intents += intent_logits.argmax(dim=1).tolist()
            gold_intents += intent.tolist()
            for path, offsets, gold in zip(model.crf.decode(emissions, mask=mask),
                                           batch['offsets'], batch['gold_spans']):
                pred_spans.append(spans_to_char(tags_to_spans([id_to_slot[i] for i in path]), offsets))
                gold_spans.append(gold_char_spans(gold))
    return (total_loss / total, intent_accuracy(pred_intents, gold_intents),
            slot_micro_f1(pred_spans, gold_spans))

---
# Part B: Stage 2 AI-enhanced system (Task 1 + Task 2)

Built after collaborating with AI to diagnose the Stage-1 baseline. It reuses Stage 1's data splits,
tokenisation, BIO alignment, word vocabulary, GloVe vectors, lemma fallback and **shared scoring functions**.

| # | Change from Stage 1 | Why |
|---|---------------------|-----|
| 1 | Character-level CNN word features concatenated with GloVe | Handles unseen / misspelled words and morphology that word vectors map to `<UNK>` |
| 2 | **Task 1:** attention pooling added to max + mean pooling for intent | Lets the intent head focus on the few decisive words |
| 3 | **Task 2:** intent-aware slot emissions (pooled sentence vector concatenated to each token state) | Slot types depend on the intent (e.g. *time* vs *date*) |
| 4 | One-cycle LR schedule (10% warm-up, cosine decay) | More stable early training |
| 5 | GloVe frozen and shared by all ensemble members | Freezing did not hurt validation scores; one shared 1.54M table frees budget for an extra member |
| 6 | Heterogeneous 3-member ensemble: intent probabilities averaged; slot emissions and CRF transitions averaged before one Viterbi decode | Ensembling diverse models reduces variance; the whole system stays under 20M parameters |

Tried and rejected on validation (see report): intent-conditioned slot-type masking, stochastic weight
averaging / longer training, smaller (128-unit) members.

## 15. Stage 2: configuration and inputs

In [10]:
torch.set_num_threads(min(8, os.cpu_count()))
MAX_CHARS = 20

S2_BASE_CFG = dict(char=True, char_filters=100, hidden=256, layers=2, dropout=0.3, attn=True,
                   intent2slot=True, lr=1e-3, weight_decay=0.01, label_smoothing=0.1,
                   word_dropout=0.25, batch_size=32, epochs=30, patience=10, grad_clip=5.0)

S2_MEMBERS = [
    dict(name='word', seed=SEED, char=False, attn=False, intent2slot=False),  # Stage-1 architecture
    dict(name='full', seed=SEED),                                           # all Stage-2 changes
    dict(name='char', seed=SEED, attn=False, intent2slot=False),            # + character CNN only
]


def add_chars(records):
    # Original-case surface form of every token, for the character CNN
    return [{**r, 'chars': [r['utt'][s:e] for s, e in r['offsets']]} for r in records]


s2_train, s2_val, s2_test = (add_chars(r) for r in (train_records, val_records, test_records))
char_to_id = {c: i + 2 for i, c in enumerate(          # 0 = pad, 1 = unknown character
    sorted({c for r in s2_train for tok in r['chars'] for c in tok}))}


class CharJointDataset(Dataset):
    def __init__(self, records):
        self.records, self.items = records, []
        for r in records:
            words = torch.tensor([token_to_id(t, word_to_id, use_lemma=True) for t in r['tokens']])
            chars = torch.zeros(len(r['chars']), MAX_CHARS, dtype=torch.long)
            for i, tok in enumerate(r['chars']):
                ids = [char_to_id.get(ch, 1) for ch in tok[:MAX_CHARS]]
                chars[i, :len(ids)] = torch.tensor(ids)
            slots = torch.tensor([slot_to_id[t] for t in r['slot_tags']])
            self.items.append((words, chars, slots, intent_to_id[r['intent']]))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        # Return the index only; collate() builds the padded batch from self.items
        return i


def make_char_loader(dataset, batch_size, shuffle, seed=SEED):
    def collate(indices):                       # dynamic padding to the longest utterance
        L, B = max(len(dataset.items[i][0]) for i in indices), len(indices)
        w = torch.zeros(B, L, dtype=torch.long)
        c = torch.zeros(B, L, MAX_CHARS, dtype=torch.long)
        s = torch.full((B, L), SLOT_PAD_ID)
        m = torch.zeros(B, L, dtype=torch.bool)
        y = torch.zeros(B, dtype=torch.long)
        for b, i in enumerate(indices):
            wi, ci, si, yi = dataset.items[i]
            n = len(wi)
            w[b, :n] = wi
            c[b, :n] = ci
            s[b, :n] = si
            m[b, :n] = True
            y[b] = yi
        return {'words': w.to(device), 'chars': c.to(device), 'slots': s.to(device),
                'mask': m.to(device), 'intent': y.to(device), 'index': indices}
    generator = torch.Generator().manual_seed(seed) if shuffle else None
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                      collate_fn=collate, generator=generator)


s2_train_ds, s2_val_ds, s2_test_ds = (CharJointDataset(r) for r in (s2_train, s2_val, s2_test))
s2_val_loader  = make_char_loader(s2_val_ds, 128, shuffle=False)
s2_test_loader = make_char_loader(s2_test_ds, 128, shuffle=False)
print("Character vocabulary:", len(char_to_id) + 2)
print("Example:", s2_train[0]['chars'], s2_train[0]['slot_tags'])

Character vocabulary: 34
Example: ['wake', 'me', 'up', 'at', 'nine', 'am', 'on', 'friday'] ['O', 'O', 'O', 'O', 'B-time', 'I-time', 'O', 'B-date']


## 16. Stage 2: model (Task 1 and Task 2)

- **Inputs:** word IDs (frozen GloVe 300d, one table shared by all members) and per-token character IDs.
- **Encoder:** `[GloVe ; CharCNN(50d chars, 100 filters, width 3, max-pool)]` → dropout →
  2-layer BiLSTM (256 per direction).
- **Task 1 output (intent head):** concat(max-pool, mean-pool, attention-pool) → linear → 60 intents.
- **Task 2 output (slot head):** each BiLSTM state concatenated with a tanh projection of the pooled sentence vector
  → linear emissions over 111 BIO tags → CRF.
- **Loss (joint):** Task 1 intent cross-entropy (label smoothing 0.1) + Task 2 CRF negative log-likelihood.
- **Optimiser:** AdamW (lr 1e-3, weight decay 0.01), one-cycle schedule, gradient clipping 5, word dropout α = 0.25.

Members switch off some components (`S2_MEMBERS`) for diversity. At inference the ensemble averages Task 1 intent
probabilities, and averages Task 2 slot emissions and CRF transition scores before a single Viterbi decode
(equivalent to adding the members' path scores).

In [11]:
class CharCNN(nn.Module):
    def __init__(self, num_chars, char_dim=50, filters=100, width=3):
        super().__init__()
        self.embedding = nn.Embedding(num_chars, char_dim, padding_idx=0)
        self.conv = nn.Conv1d(char_dim, filters, width, padding=width // 2)

    def forward(self, chars):                                   # (B, L, C)
        B, L, C = chars.shape
        flat = chars.view(B * L, C)
        x = self.conv(self.embedding(flat).transpose(1, 2))     # (B*L, filters, C)
        x = x.masked_fill((flat == 0).unsqueeze(1), -1e4).max(dim=2).values
        word_exists = (chars[:, :, 0] != 0).unsqueeze(-1)
        return x.clamp(min=-1e3).view(B, L, -1) * word_exists


class JointBiLSTMv2(nn.Module):
    def __init__(self, embedding, cfg):
        super().__init__()
        self.cfg = cfg
        self.embedding = embedding                  # shared, frozen GloVe table
        in_dim = embedding.embedding_dim
        if cfg['char']:
            self.char_cnn = CharCNN(len(char_to_id) + 2, filters=cfg['char_filters'])
            in_dim += cfg['char_filters']
        H = cfg['hidden']
        self.lstm = nn.LSTM(in_dim, H, cfg['layers'], batch_first=True,
                            bidirectional=True, dropout=cfg['dropout'])
        self.dropout = nn.Dropout(cfg['dropout'])
        pool_dim = 4 * H
        if cfg['attn']:
            self.attention = nn.Linear(2 * H, 1)
            pool_dim += 2 * H
        self.intent_classifier = nn.Linear(pool_dim, num_classes)
        slot_in = 2 * H
        if cfg['intent2slot']:
            self.intent_to_slot = nn.Linear(pool_dim, 2 * H)
            slot_in += 2 * H
        self.slot_classifier = nn.Linear(slot_in, num_slot_labels)
        self.crf = CRF(num_slot_labels, batch_first=True)

    def forward(self, words, chars, mask):
        x = self.embedding(words)
        if self.cfg['char']:
            x = torch.cat([x, self.char_cnn(chars)], dim=-1)
        x = self.dropout(x)
        packed = pack_padded_sequence(x, mask.sum(1).cpu(), batch_first=True, enforce_sorted=False)
        h, _ = self.lstm(packed)
        h, _ = pad_packed_sequence(h, batch_first=True, total_length=words.size(1))
        h = self.dropout(h)
        m = mask.unsqueeze(-1)
        pools = [h.masked_fill(~m, -1e4).max(dim=1).values, (h * m).sum(1) / m.sum(1)]
        if self.cfg['attn']:
            weights = self.attention(h).masked_fill(~m, -1e4).softmax(dim=1)
            pools.append((weights * h).sum(1))
        sentence = torch.cat(pools, dim=-1)
        intent_logits = self.intent_classifier(sentence)
        if self.cfg['intent2slot']:
            context = torch.tanh(self.intent_to_slot(sentence))
            h = torch.cat([h, context.unsqueeze(1).expand(-1, h.size(1), -1)], dim=-1)
        return intent_logits, self.slot_classifier(h)


shared_embedding = nn.Embedding.from_pretrained(
    torch.as_tensor(build_embedding_matrix(word_to_id)), freeze=True, padding_idx=PAD_ID).to(device)


def build_member(cfg):
    torch.manual_seed(cfg['seed'])
    return JointBiLSTMv2(shared_embedding, cfg).to(device)


def count_params(models):
    # Shared parameters (the GloVe table) are counted once for the whole system
    unique = {id(p): p for m in models for p in m.parameters()}.values()
    return sum(p.numel() for p in unique), sum(p.numel() for p in unique if p.requires_grad)


_preview = [build_member(dict(S2_BASE_CFG, **m)) for m in S2_MEMBERS]
for m, model in zip(S2_MEMBERS, _preview):
    print(f"{m['name']:>8}: {count_params([model])[1]:,} trainable parameters")
print(f"Shared frozen GloVe table: {shared_embedding.weight.numel():,} parameters")
S2_TOTAL, S2_TRAINABLE = count_params(_preview)
print(f"Total parameters (whole ensemble, incl. word vectors): {S2_TOTAL:,}")
print(f"Total trainable parameters:                            {S2_TRAINABLE:,}")
assert S2_TOTAL <= 20_000_000, "parameter budget exceeded"
print(f"Within budget: {100 * S2_TOTAL / 20_000_000:.1f}% of 20M")
del _preview

    word: 2,850,730 trainable parameters
    full: 3,947,339 trainable parameters
    char: 3,072,330 trainable parameters
Shared frozen GloVe table: 1,543,800 parameters
Total parameters (whole ensemble, incl. word vectors): 11,414,199
Total trainable parameters:                            9,870,399
Within budget: 57.1% of 20M


## 17. Stage 2: loss, ensemble inference and evaluation (shared scoring functions)

In [12]:
s2_word_freq = word_frequencies(train_records, word_to_id)


def joint_loss(model, intent_logits, emissions, batch, label_smoothing):
    return F.cross_entropy(intent_logits, batch['intent'], label_smoothing=label_smoothing) \
           - model.crf(emissions, batch['slots'].clamp(min=0), mask=batch['mask'], reduction='mean')


def validation_loss(model, loader):
    model.eval()
    total, n = 0.0, 0
    with torch.no_grad():
        for b in loader:
            il, em = model(b['words'], b['chars'], b['mask'])
            total += joint_loss(model, il, em, b, 0.0).item() * len(b['intent'])
            n += len(b['intent'])
    return total / n


def predict_ensemble(models, loader, records):
    """Returns (intent labels, predicted character spans, intent probabilities)."""
    for m in models:
        m.eval()
    k = len(models)
    crf = models[0].crf
    pred_intents, pred_spans, all_probs = [], [], []
    with torch.no_grad():
        for b in loader:
            # Sum the members' intent probabilities, slot emissions and CRF transition scores
            probs = emissions = trans = start = end = 0
            for m in models:
                il, em = m(b['words'], b['chars'], b['mask'])
                probs = probs + il.softmax(-1)
                emissions = emissions + em
                trans = trans + m.crf.transitions
                start = start + m.crf.start_transitions
                end = end + m.crf.end_transitions

            # Decode once with the averaged CRF, then restore the first member's CRF
            saved_trans = crf.transitions.data.clone()
            saved_start = crf.start_transitions.data.clone()
            saved_end = crf.end_transitions.data.clone()
            crf.transitions.data = trans / k
            crf.start_transitions.data = start / k
            crf.end_transitions.data = end / k
            paths = crf.decode(emissions / k, mask=b['mask'])
            crf.transitions.data = saved_trans
            crf.start_transitions.data = saved_start
            crf.end_transitions.data = saved_end

            for i, path, p in zip(b['index'], paths, probs / k):
                pred_intents.append(intents[int(p.argmax())])
                pred_spans.append(spans_to_char(tags_to_spans([id_to_slot[t] for t in path]),
                                                records[i]['offsets']))
                all_probs.append(p)
    return pred_intents, pred_spans, torch.stack(all_probs)


def evaluate_ensemble(models, loader, records):
    pi, ps, _ = predict_ensemble(models, loader, records)
    acc = intent_accuracy(pi, [r['intent'] for r in records])
    f1 = slot_micro_f1(ps, [gold_char_spans(r['gold_spans']) for r in records])
    return {'intent_acc': acc, 'slot_f1': f1, 'combined': (acc + f1) / 2}

## 21. Stage 2: final evaluation of the checkpoint (validation and test splits)

Reload the saved checkpoint into freshly built models (no further training), check the validation score
matches the one above, then evaluate once on the **test split**.
Both Task 1 (intent accuracy) and Task 2 (slot micro-F1) are reported.

In [13]:
# Normally set in section 20 (skipped, it trains and saves the model)
S2_CHECKPOINT = os.path.join("model", "stage2_ensemble_best.pt")

In [14]:
ckpt = torch.load(S2_CHECKPOINT)
loaded = []
for cfg, state in zip(ckpt['members'], ckpt['states']):
    model = build_member(cfg)
    model.load_state_dict(state)
    loaded.append(model.eval())
s2_total, s2_trainable = count_params(loaded)
print(f"Loaded {len(loaded)} members | total parameters {s2_total:,} | trainable {s2_trainable:,}")

s2_val_scores = evaluate_ensemble(loaded, s2_val_loader, s2_val)
print(f"Validation (reloaded): intent accuracy {s2_val_scores['intent_acc']:.2f}% | "
      f"slot micro-F1 {s2_val_scores['slot_f1']:.2f}%")
assert abs(s2_val_scores['combined'] - ckpt['val_scores']['combined']) < 1e-9, \
    "reloaded validation score differs from the saved one"
print("Matches the saved best validation score.")

s2_test_intents, s2_test_spans, s2_test_probs = predict_ensemble(loaded, s2_test_loader, s2_test)
s2_test_acc = intent_accuracy(s2_test_intents, [r['intent'] for r in s2_test])
s2_test_f1 = slot_micro_f1(s2_test_spans, [gold_char_spans(r['gold_spans']) for r in s2_test])
print(f"\nTEST: intent accuracy {s2_test_acc:.2f}% | slot micro-F1 {s2_test_f1:.2f}%")

Loaded 3 members | total parameters 11,414,199 | trainable 9,870,399


Validation (reloaded): intent accuracy 87.75% | slot micro-F1 82.72%
Matches the saved best validation score.



TEST: intent accuracy 86.82% | slot micro-F1 80.75%


## 22. Results: Stage 1 vs Stage 2 (report table) and Stage-2 error analysis

The error analysis lists Task 1 (intent) errors first, then Task 2 (slot) errors, on the test split.

In [15]:
intent_errors = [(r['utt'], r['intent'], p, float(prob.max()))
                 for r, p, prob in zip(s2_test, s2_test_intents, s2_test_probs) if p != r['intent']]
print(f"Intent errors: {len(intent_errors)} / {len(s2_test)}")
confusions = collections.Counter((g, p) for _, g, p, _ in intent_errors)
print("\nMost frequent intent confusions (gold -> predicted):")
for (g, p), n in confusions.most_common(10):
    print(f"  {n:3d}  {g} -> {p}")
print("\nSample intent errors:")
for utt, g, p, conf in intent_errors[:8]:
    print(f"  {utt!r}\n      gold {g} | pred {p} (p={conf:.2f})")

def format_spans(spans, utt):
    # Show (type, start, end) spans as readable "[type: text]" strings
    return sorted(f"[{t}: {utt[s:e]}]" for t, s, e in spans)


slot_errors = []
type_fn, type_fp = collections.Counter(), collections.Counter()
for r, pred in zip(s2_test, s2_test_spans):
    gold = gold_char_spans(r['gold_spans'])
    for slot_type, start, end in gold - pred:
        type_fn[slot_type] += 1
    for slot_type, start, end in pred - gold:
        type_fp[slot_type] += 1
    if gold != pred:
        slot_errors.append((r['utt'], format_spans(gold - pred, r['utt']),
                            format_spans(pred - gold, r['utt'])))
print(f"\nUtterances with at least one slot error: {len(slot_errors)} / {len(s2_test)}")
print("Slot types with most missed (FN) / spurious (FP) spans:")
type_errors = type_fn + type_fp
for t, _ in type_errors.most_common(10):
    print(f"  {t:<20} FN {type_fn[t]:3d} | FP {type_fp[t]:3d}")
print("\nSample slot errors:")
for utt, missed, spurious in slot_errors[:8]:
    print(f"  {utt!r}\n      missed {missed} | spurious {spurious}")

Intent errors: 392 / 2974

Most frequent intent confusions (gold -> predicted):
   14  general_quirky -> qa_factoid
   13  qa_factoid -> general_quirky
   10  general_quirky -> calendar_query
    8  news_query -> general_quirky
    6  general_quirky -> news_query
    6  calendar_set -> calendar_query
    6  qa_definition -> general_quirky
    5  general_quirky -> weather_query
    5  recommendation_movies -> general_quirky
    5  play_radio -> play_music

Sample intent errors:
  'pink is all we need'
      gold iot_hue_lightchange | pred play_music (p=0.36)
  'and the darkness has fallen'
      gold iot_hue_lighton | pred general_quirky (p=0.18)
  "it's dirty here make some noise"
      gold iot_cleaning | pred play_music (p=0.76)
  'hoover the hallway'
      gold iot_cleaning | pred email_querycontact (p=0.36)
  'i want the status on my screen brightness'
      gold general_quirky | pred iot_hue_lightdim (p=0.27)
  'send a giggle my way'
      gold general_joke | pred general_quirky (

## 23. Hypothesis checks (Stage-2 test predictions)
Checks the hypotheses from the data analysis notebook against the Stage-2 test predictions.

In [16]:
# Hypothesis checks on the Stage-2 test predictions (uses s2_test, s2_test_intents, s2_test_spans)
def scenario_of(intent):
    # Every MASSIVE intent name starts with its scenario, e.g. iot_hue_lighton -> iot
    return intent.split('_')[0]


def is_unseen(token):
    # The model only sees this token as <UNK> (not in the training vocab, even after the lemma fallback)
    return token_to_id(token, word_to_id, use_lemma=True) == UNK_ID


def length_bucket(n):
    for upper, label in [(3, '1-3'), (5, '4-5'), (8, '6-8'), (12, '9-12')]:
        if n <= upper:
            return label
    return '13+'


def f1_from_counts(tp, fp, fn):
    return 0.0 if tp == 0 else 100 * 2 * tp / (2 * tp + fp + fn)


s2_gold_spans = [gold_char_spans(r['gold_spans']) for r in s2_test]
error_df = pd.DataFrame({
    'utt': [r['utt'] for r in s2_test],
    'gold_intent': [r['intent'] for r in s2_test],
    'pred_intent': s2_test_intents,
    'n_tokens': [len(r['tokens']) for r in s2_test],
    'has_unseen': [any(is_unseen(t) for t in r['tokens']) for r in s2_test],
    'tp': [len(g & p) for g, p in zip(s2_gold_spans, s2_test_spans)],
    'fp': [len(p - g) for g, p in zip(s2_gold_spans, s2_test_spans)],
    'fn': [len(g - p) for g, p in zip(s2_gold_spans, s2_test_spans)],
})
error_df['intent_ok'] = error_df['gold_intent'] == error_df['pred_intent']
error_df['length'] = error_df['n_tokens'].map(length_bucket)

# Same overall scores as the shared scoring functions
assert abs(100 * error_df['intent_ok'].mean() - s2_test_acc) < 1e-9
assert abs(f1_from_counts(error_df['tp'].sum(), error_df['fp'].sum(), error_df['fn'].sum()) - s2_test_f1) < 1e-9


def breakdown(by):
    # Intent accuracy and slot micro-F1 per group of test sentences
    g = error_df.groupby(by, observed=True)
    return pd.DataFrame({
        'sentences': g.size(),
        'intent acc (%)': (100 * g['intent_ok'].mean()).round(2),
        'slot F1 (%)': g.apply(lambda x: round(f1_from_counts(x['tp'].sum(), x['fp'].sum(), x['fn'].sum()), 2)),
    })


print(f"Test sentences: {len(error_df)} | intent acc {s2_test_acc:.2f}% | slot F1 {s2_test_f1:.2f}% (matches above)")


Test sentences: 2974 | intent acc 86.82% | slot F1 80.75% (matches above)


### H1: Are most intent errors between sibling intents?

In [17]:
# H1: are most intent errors between sibling intents (same scenario)?
wrong = error_df[~error_df['intent_ok']]
same = wrong['gold_intent'].map(scenario_of) == wrong['pred_intent'].map(scenario_of)
quirky = (wrong['gold_intent'] == 'general_quirky') | (wrong['pred_intent'] == 'general_quirky')

print(f"Intent errors: {len(wrong)}")
print(f"  same scenario (siblings): {same.sum()} ({100 * same.mean():.2f}%)")
print(f"  different scenario:       {(~same).sum()} ({100 * (~same).mean():.2f}%)")
print(f"  involving general_quirky: {quirky.sum()} ({100 * quirky.mean():.2f}%)")

print("\nLowest intent accuracy by scenario:")
print(breakdown(error_df['gold_intent'].map(scenario_of).rename('scenario')).sort_values('intent acc (%)').head(8))


Intent errors: 392
  same scenario (siblings): 94 (23.98%)
  different scenario:       298 (76.02%)
  involving general_quirky: 142 (36.22%)

Lowest intent accuracy by scenario:
                sentences  intent acc (%)  slot F1 (%)
scenario                                              
general               189           65.08        75.71
music                  81           67.90        57.14
takeaway               57           80.70        78.26
news                  124           83.06        75.34
recommendation         94           84.04        68.04
qa                    288           85.42        81.28
audio                  62           85.48        96.00
transport             124           85.48        89.01


**Hypothesis is wrong.**
- Only 23.98% of intent errors are between siblings (94 / 392).
- 36.22% involve general_quirky. The general scenario is the worst (65.08% intent acc).
- So the catch-all intent is the bigger problem, not siblings.

### H2: Are rare intents predicted as a common sibling?

In [18]:
# H2: are rare intents (25 or fewer training examples) predicted as a common sibling?
train_intent_counts = collections.Counter(r['intent'] for r in train_records)
rare = error_df[error_df['gold_intent'].map(train_intent_counts) <= 25].copy()
rare['train examples'] = rare['gold_intent'].map(train_intent_counts)
rare['same scenario'] = rare['gold_intent'].map(scenario_of) == rare['pred_intent'].map(scenario_of)

print(f"Rare-intent test sentences: {len(rare)} | correct: {rare['intent_ok'].sum()}")
rare[['utt', 'gold_intent', 'train examples', 'pred_intent', 'intent_ok', 'same scenario']]


Rare-intent test sentences: 14 | correct: 8


,utt,gold_intent,train examples,pred_intent,intent_ok,same scenario
3,and the darkness has fallen,iot_hue_lighton,22,general_quirky,False,False
8,what's up olly,general_greet,25,general_greet,True,True
51,light up the lights in the kitchen,iot_hue_lighton,22,iot_hue_lightup,False,True
184,where is volume setting,audio_volume_other,18,audio_volume_other,True,True
187,do not play rock metal,music_dislikeness,14,play_music,False,False
340,i never want to hear that song again,music_dislikeness,14,play_music,False,False
507,i hate this song,music_dislikeness,14,music_likeness,False,True
522,don't play this song again,music_dislikeness,14,music_dislikeness,True,True
596,please turn on my balcony lights,iot_hue_lighton,22,iot_hue_lighton,True,True
712,change volume to thirty five percent,audio_volume_other,18,audio_volume_up,False,True


**Hypothesis is partly correct.**
- Rare intents get 8 / 14 correct. 3 of the wrong ones go to a sibling, 3 don't.
- "do not play rock metal" -> play_music. The model sees "play" and ignores the "not".

### H3: Are sentences with unseen words harder?

In [19]:
# H3: are sentences with unseen words (still <UNK> after the lemma fallback) harder?
breakdown('has_unseen')


,sentences,intent acc (%),slot F1 (%)
has_unseen,,,
False,2503,89.05,83.71
True,471,74.95,67.20


**Hypothesis is correct.**
- With unseen words: intent 74.95%, slot F1 67.20%. Without: 89.05%, 83.71%.
- About 14 points lower for intent and 17 for slots. Biggest drop of all the checks.

### H4: Which slot types have the lowest F1?

In [20]:
# H4: which slot types have the most errors? (F1 per type, not just error counts)
slot_tp, slot_fp, slot_fn = collections.Counter(), collections.Counter(), collections.Counter()
for g, p in zip(s2_gold_spans, s2_test_spans):
    for t, _, _ in g & p:
        slot_tp[t] += 1
    for t, _, _ in p - g:
        slot_fp[t] += 1
    for t, _, _ in g - p:
        slot_fn[t] += 1

slot_types = sorted(set(slot_tp) | set(slot_fp) | set(slot_fn))
per_type = pd.DataFrame({
    'test spans': [slot_tp[t] + slot_fn[t] for t in slot_types],
    'missed (FN)': [slot_fn[t] for t in slot_types],
    'spurious (FP)': [slot_fp[t] for t in slot_types],
    'F1 (%)': [round(f1_from_counts(slot_tp[t], slot_fp[t], slot_fn[t]), 2) for t in slot_types],
}, index=slot_types)

print("Slot types with at least 50 test spans, lowest F1 first:")
per_type[per_type['test spans'] >= 50].sort_values('F1 (%)')


Slot types with at least 50 test spans, lowest F1 first:


,test spans,missed (FN),spurious (FP),F1 (%)
food_type,72,17,31,69.62
weather_descriptor,82,18,24,75.29
event_name,260,66,58,75.78
list_name,60,15,12,76.92
business_name,92,21,20,77.60
music_genre,50,9,12,79.61
time,193,38,36,80.73
artist_name,61,8,16,81.54
place_name,281,53,50,81.57
device_type,57,8,13,82.35


**Hypothesis is partly correct.**
- Time is low (80.73%) but person is fine (86.47%).
- Worst are food_type (69.62%), weather_descriptor, event_name, list_name and business_name.
- date is one of the best (90.95%).

### H5: What kind of slot errors are they?

In [21]:
# H5: what kind of slot errors are they?
def classify_slot_errors(gold, pred):
    # Each missed gold span: type swap, boundary, type + boundary, or missed.
    # Each extra prediction not overlapping any gold span: spurious.
    kinds, wrong_pred = [], pred - gold
    for gt, gs, ge in gold - pred:
        overlap = [(pt, ps, pe) for pt, ps, pe in wrong_pred if ps < ge and pe > gs]
        if any((ps, pe) == (gs, ge) for _, ps, pe in overlap):
            kinds.append((gt, 'type swap'))
        elif any(pt == gt for pt, _, _ in overlap):
            kinds.append((gt, 'boundary'))
        elif overlap:
            kinds.append((gt, 'type + boundary'))
        else:
            kinds.append((gt, 'missed'))
    for pt, ps, pe in wrong_pred:
        if not any(ps < ge and pe > gs for _, gs, ge in gold):
            kinds.append((pt, 'spurious'))
    return kinds


slot_errors_df = pd.DataFrame(
    [k for g, p in zip(s2_gold_spans, s2_test_spans) for k in classify_slot_errors(g, p)],
    columns=['slot type', 'kind'])
print("All slot errors by kind:")
print(slot_errors_df['kind'].value_counts())
print("\ndate / time errors by kind:")
print(pd.crosstab(slot_errors_df['slot type'], slot_errors_df['kind']).reindex(['date', 'time']).fillna(0).astype(int))

swaps = collections.Counter()
for g, p in zip(s2_gold_spans, s2_test_spans):
    for gt, gs, ge in g - p:
        for pt, ps, pe in p - g:
            if (ps, pe) == (gs, ge):
                swaps[(gt, pt)] += 1
print("\nMost common type swaps (gold -> predicted):")
for (gt, pt), n in swaps.most_common(8):
    print(f"  {n:3d}  {gt} -> {pt}")


All slot errors by kind:
kind
missed             246
spurious           209
boundary           152
type swap           94
type + boundary     54
Name: count, dtype: int64

date / time errors by kind:
kind       boundary  missed  spurious  type + boundary  type swap
slot type                                                        
date             17      10        12                0          2
time             15      12        10                4          7

Most common type swaps (gold -> predicted):
    8  place_name -> currency_name
    6  time -> date
    3  news_topic -> place_name
    3  ingredient -> food_type
    3  person -> podcast_name
    2  song_name -> time
    2  music_genre -> song_name
    2  news_topic -> person


**Hypothesis is correct.**
- Boundary errors are the most common for date (17) and time (15).
- date/time swaps are rare (time -> date only 6).
- Overall most slot errors are missed (246) or spurious (209).

### H6: Does performance drop for very short or very long sentences?

In [22]:
# H6: does performance drop for very short or very long sentences?
print(breakdown('length').reindex(['1-3', '4-5', '6-8', '9-12', '13+']))
print("\nLongest test sentences:")
error_df.sort_values('n_tokens', ascending=False).head(3)[
    ['n_tokens', 'utt', 'gold_intent', 'pred_intent', 'intent_ok', 'fp', 'fn']]


        sentences  intent acc (%)  slot F1 (%)
length                                        
1-3           344           75.87        79.67
4-5           708           87.85        79.37
6-8          1110           88.65        83.56
9-12          599           88.31        78.91
13+           213           87.32        79.39

Longest test sentences:


,n_tokens,utt,gold_intent,pred_intent,intent_ok,fp,fn
2042,61,both young drivers and provisional drivers cra...,recommendation_events,general_quirky,False,0,0
2553,33,what is a neutral density filter and would i w...,qa_factoid,general_quirky,False,0,0
2293,28,i want it to be able to tell me statistics abo...,general_quirky,general_quirky,True,0,0


**The problem stems from short sentences, but only for intent.**
- 1-3 tokens: 75.87% intent, about 12 points lower than the other lengths.
- Their slot F1 (79.67%) is not lower than the other lengths, so short sentences only hurt intent.
- Long sentences are fine (13+: 87.32% intent).
- The 61-token sentence is still wrong (predicted general_quirky).

### H7: Do scenario-specific slot types score higher than date / time / place_name / person?

In [23]:
# H7 (partial check): do scenario-specific slot types score higher than date / time / place_name / person?
# A full test of the intent-aware slot head would need the same model trained without it.
train_slot_scenarios = collections.defaultdict(collections.Counter)
for r in train_records:
    for s in r['gold_spans']:
        train_slot_scenarios[s['type']][scenario_of(r['intent'])] += 1
specific = {t for t, c in train_slot_scenarios.items() if max(c.values()) / sum(c.values()) >= 0.9}
common = {'date', 'time', 'place_name', 'person'}
print(f"Scenario-specific slot types (90%+ of training spans in one scenario): {len(specific)}")

for name, group in [('scenario-specific', specific), ('date / time / place_name / person', common)]:
    tp_, fp_, fn_ = (sum(c[t] for t in group) for c in (slot_tp, slot_fp, slot_fn))
    print(f"  {name:<34} slot F1 {f1_from_counts(tp_, fp_, fn_):.2f}% over {tp_ + fn_} test spans")


Scenario-specific slot types (90%+ of training spans in one scenario): 33
  scenario-specific                  slot F1 75.79% over 735 test spans
  date / time / place_name / person  slot F1 85.97% over 1109 test spans


**Hypothesis is wrong.**
- Scenario-specific slot types score lower (75.79%) than date/time/place_name/person (85.97%).
- Probably because they're rarer, so less training data.
- Only a partial check. A proper test needs the model trained without the intent-aware slot part.